In [1]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
import cloudvolume
from tqdm import tqdm
import glob
import os

# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [2]:
# Synapses df
in_tag = client.materialize.query_table('excitatory_neuron_label')
exn_tag = client.materialize.query_table('spinalcord_neuron_clusters')
sensory_tag=client.materialize.query_table('sensory_neuron_label')

Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.
Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


In [3]:
# counts per tag (sorted descending)
exn_tag['tag'].value_counts()


tag
v1               41
v2               36
np_module        27
cltmr_module     24
r                24
p2_poly          18
adltmr_module    15
p2_noci          14
noci_module      10
p2_cold           8
p1                6
Name: count, dtype: int64

In [4]:
# tag -> list of pt_root_id values
tag_to_root_ids = (
    exn_tag.groupby('tag')['pt_root_id']
    .apply(list)
)

tag_to_root_ids_dict = tag_to_root_ids.to_dict()
tag_to_root_ids_dict



{'adltmr_module': [720575940927248352,
  720575940937862061,
  720575940878312571,
  720575940895289006,
  720575940888165967,
  720575940886853257,
  720575940890900801,
  720575940948987670,
  720575940907584340,
  720575940883729656,
  720575940896732090,
  720575940902624978,
  720575940941124460,
  720575940917422227,
  720575940880839442],
 'cltmr_module': [720575940937848237,
  720575940891183798,
  720575940924182661,
  720575940900956101,
  720575940902703006,
  720575940861450226,
  720575940922814762,
  720575940878978356,
  720575940893177002,
  720575940883197911,
  720575940878956340,
  720575940912929024,
  720575940890245126,
  720575940930905464,
  720575940940380075,
  720575940908781145,
  720575940881620931,
  720575940891181902,
  720575940889238903,
  720575940899283650,
  720575940880875929,
  720575940901384751,
  720575940918316341,
  720575940899281090],
 'noci_module': [720575940941106028,
  720575940922949405,
  720575940899282056,
  720575940917429907,
  72

In [5]:
exn_tag

,id,created,superceded_id,valid,tag,pt_supervoxel_id,pt_root_id,pt_position
0,2,2025-09-20 03:19:48.176531+00:00,NaN,t,p2_noci,77053912649181423,720575940865160474,"[22364, 2792, 812]"
1,3,2025-09-20 03:19:48.177172+00:00,NaN,t,p2_noci,80853826042750294,720575940881844049,"[36293, 2701, 3171]"
2,4,2025-09-20 03:19:48.177858+00:00,NaN,t,p2_noci,82824356568335590,720575940902401208,"[43341, 3453, 2176]"
3,5,2025-09-20 03:19:48.178508+00:00,NaN,t,p2_noci,72479876497660276,720575940896702138,"[5689, 2462, 2592]"
4,6,2025-09-20 03:19:48.179124+00:00,NaN,t,p2_noci,78953868540584103,720575940898432765,"[29367, 2807, 368]"
...,...,...,...,...,...,...,...,...
218,219,2025-09-20 03:19:48.307032+00:00,NaN,t,r,78180156623161798,720575940894664404,"[26383, 3997, 1701]"
219,220,2025-09-20 03:19:48.307639+00:00,NaN,t,r,78109650440024886,720575940903415565,"[26118, 3477, 1701]"
220,221,2025-09-20 03:19:48.308194+00:00,NaN,t,r,77476263022984897,720575940875854397,"[23885, 3254, 1751]"
221,222,2025-09-20 03:19:48.308781+00:00,NaN,t,r,77898750500149038,720575940910295881,"[25559, 4188, 2001]"


In [9]:
import re
import numpy as np
import pandas as pd

def extract_y(pos):
    # 1) If it's already array-like, try to index safely
    if isinstance(pos, (list, tuple, np.ndarray)):
        arr = np.asarray(pos)
        arr = arr.reshape(-1)  # handles (1,3), (3,), etc.
        if arr.size >= 2:
            return arr[1]

    # 2) If it's a string, pull out numbers robustly
    if isinstance(pos, str):
        nums = re.findall(r'-?\d+\.?\d*', pos)  # ints/floats, optional minus
        if len(nums) >= 2:
            return float(nums[1])

    return np.nan

tmp = exn_tag.copy()
tmp["y"] = tmp["pt_position"].apply(extract_y)
tmp["y"] = pd.to_numeric(tmp["y"], errors="coerce")

# Show the 5 rows with the highest y
top5 = tmp.nlargest(5, "y")[["tag", "pt_root_id", "pt_position", "y"]]
top5




,tag,pt_root_id,pt_position,y
121,adltmr_module,720575940917422227,"[23227, 5435, 2051]",5435
120,adltmr_module,720575940941124460,"[25705, 5363, 1401]",5363
122,adltmr_module,720575940880839442,"[24788, 5284, 1701]",5284
119,adltmr_module,720575940902624978,"[26000, 5201, 1651]",5201
118,adltmr_module,720575940896732090,"[23485, 5196, 1401]",5196


In [11]:
targets = [
    720575940903996112,
    720575940888148592,
    720575940899782543,
    720575940900300153
]

# if pt_root_id might be stored as strings, normalize to int first
pt_ids = pd.to_numeric(exn_tag['pt_root_id'], errors='coerce')

mask = pt_ids.isin(targets)

# True/False for each target
found_map = {tid: bool((pt_ids == tid).any()) for tid in targets}
found_map


{720575940903996112: False,
 720575940888148592: False,
 720575940899782543: False,
 720575940900300153: False}